In [0]:
# Databricks notebook source
# Run once. Re-running is non-destructive.
CATALOG = 'github_health'
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
for schema in ['raw', 'ops', 'bronze', 'silver']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {CATALOG}.ops.pipeline_execution_logs (
 execution_id STRING NOT NULL, batch_id STRING NOT NULL, notebook_name STRING,
 layer STRING, run_date DATE, load_type STRING, start_time TIMESTAMP,
 end_time TIMESTAMP, status STRING, rows_processed BIGINT, error_message STRING
) USING DELTA''')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {CATALOG}.ops.pipeline_watermarks (
 repository STRING NOT NULL, entity STRING NOT NULL, layer STRING NOT NULL,
 last_successful_watermark TIMESTAMP, batch_id STRING, updated_at TIMESTAMP
) USING DELTA''')
for layer in ('bronze','silver'):
    spark.sql(f'''CREATE TABLE IF NOT EXISTS {CATALOG}.ops.{layer}_quarantine (
      quarantine_id STRING NOT NULL, batch_id STRING, repository STRING,
      entity STRING, { 'source_file STRING,' if layer == 'bronze' else 'record_key STRING,' }
      raw_payload STRING, error_type STRING, error_reason STRING,
      load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
    ) USING DELTA''')
print('Operations tables available')
for name in ['pipeline_execution_logs','pipeline_watermarks','bronze_quarantine','silver_quarantine']:
    assert spark.catalog.tableExists(f'{CATALOG}.ops.{name}'), name

# Forward-compatible additive migration for detailed processing metrics.
log_name=f'{CATALOG}.ops.pipeline_execution_logs'
existing={f.name for f in spark.table(log_name).schema.fields}
new_fields=[('upserted_rows','BIGINT'),('quarantined_rows','BIGINT')]
missing=[f'{name} {dtype}' for name,dtype in new_fields if name not in existing]
if missing:
    spark.sql(f"ALTER TABLE {log_name} ADD COLUMNS ({', '.join(missing)})")
print('Audit metric columns ready')


In [0]:
%sql

SHOW TABLES IN github_health.ops;
